# Stretch — build an automated cleaning pipeline

**When:** you finished the main cleaning exercise, or as take-home.

**Why:** You just spent 30 minutes cleaning a messy reactions log by hand. Next week your PI hands you another log in the same format from a different lab. Do you want to spend another 30 minutes doing it again?

The Data Stewardship lecture argued that **data + code + docs is a triad**. What you built in the main tutorial was the data leg. This notebook builds the code leg: a reusable pipeline with function signatures, tests, and a second messy dataset you can generalize to.

If both datasets flow through your code and land as clean tidy tables — you have built a genuine institutional asset.

## Setup — install packages

Run this once per Colab session (also needed if you're running locally without these installed).

In [ ]:
%pip install -q pandas openpyxl

## How this notebook works

You'll fill in 6 functions, one at a time. Each function has:
1. A **markdown cell** describing what it should do and giving examples.
2. A **code cell** where you write the function. It starts as a stub that raises `NotImplementedError`.
3. A **test cell** that immediately checks your implementation against expected behavior. Run it — if it prints `✓ passed`, you're good; if it prints `✗ FAILED` (or errors out), fix your function and re-run.

Do them in order. Each function builds on the previous ones.

## 0. Setup — upload your files

On Colab: click the **Files** icon in the left sidebar, then **Upload to session storage**. Upload:
1. `sprint-dataset-messy.xlsx` (the tutorial messy dataset)
2. `new-experiments-messy.xlsx` (the second messy dataset — see the stretch/ folder)
3. `catalyst_reference_tidy.csv` (the gold-standard catalyst reference)
4. `solvent_reference_tidy.csv` (the gold-standard solvent reference)
5. `reactions_log_tidy.csv` (the gold-standard clean reactions log — used only in the end-to-end test)

If you're running this locally in Jupyter instead of Colab, put those files in the same folder as this notebook.

Run the cell below to import libraries and load the reference CSVs.

In [ ]:
import pandas as pd
from pathlib import Path

# On Colab, files uploaded to session storage land in /content/
# Locally, they should be in the notebook's own folder.
# The Path() below tries both.

def _find(name: str) -> Path:
    for candidate in [Path(name), Path("/content") / name, Path.cwd() / name]:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"could not find {name} — did you upload it?")

CATALYST_REF = pd.read_csv(_find("catalyst_reference_tidy.csv"))
SOLVENT_REF  = pd.read_csv(_find("solvent_reference_tidy.csv"))

print("Catalyst reference:")
display(CATALYST_REF)
print("\nSolvent reference:")
display(SOLVENT_REF)

## 1. `canonicalize_catalyst_names(series)`

Map free-typed catalyst names in a pandas Series to their canonical form.

**Canonical values** come from `catalyst_reference_tidy.csv` (loaded above as `CATALYST_REF`).

**Examples of the mapping:**

| Input variant                              | Canonical output |
|--------------------------------------------|------------------|
| `'pd(pph3)4'`, `'Tetrakis Pd'`, `'PdPPh3_4'` | `'Pd(PPh3)4'`    |
| `'PdCl2·dppf'`, `'Pd(dppf)Cl2'`            | `'PdCl2(dppf)'`  |
| `'ni(cod)2'`, `'Ni COD 2'`, `'Nickel COD'` | `'Ni(cod)2'`     |

**Fail loud on unknowns** — raise `ValueError` if a value doesn't match. Silent-pass is exactly the bug you're trying to prevent.

**Approach:** build a dictionary mapping every known variant (lowercased, whitespace stripped) to its canonical form. Look each value up. Raise on miss.

In [ ]:
def canonicalize_catalyst_names(series: pd.Series) -> pd.Series:
    """Map free-typed catalyst names to their canonical form."""
    # TODO: your implementation goes here
    raise NotImplementedError


In [ ]:
# Test cell — run to check your implementation
try:
    out = canonicalize_catalyst_names(pd.Series(["pd(pph3)4", "Tetrakis Pd", "PdPPh3_4"]))
    assert (out == "Pd(PPh3)4").all(), f"expected all Pd(PPh3)4, got {list(out)}"
    print("✓ mapping variants works")

    try:
        canonicalize_catalyst_names(pd.Series(["unknown catalyst xyz"]))
        print("✗ FAILED: should have raised ValueError on unknown catalyst")
    except ValueError:
        print("✓ raises ValueError on unknown")
    except NotImplementedError:
        raise

    print("\n✓ passed")
except NotImplementedError:
    print("(not yet implemented)")
except AssertionError as e:
    print(f"✗ FAILED: {e}")


## 2. `canonicalize_solvent_names(series)`

Same shape as the catalyst version. Canonical values come from `solvent_reference_tidy.csv` (`SOLVENT_REF`).

**Test:** `'acetonitrile'`, `'MeCN'`, `'ACN'`, `'CH3CN'` all map to `'MeCN'`.

In [ ]:
def canonicalize_solvent_names(series: pd.Series) -> pd.Series:
    """Map free-typed solvent names to their canonical form."""
    # TODO: your implementation goes here
    raise NotImplementedError


In [ ]:
# Test cell — run to check your implementation
try:
    out = canonicalize_solvent_names(pd.Series(["acetonitrile", "MeCN", "ACN", "CH3CN"]))
    assert (out == "MeCN").all(), f"expected all MeCN, got {list(out)}"
    print("✓ passed")
except NotImplementedError:
    print("(not yet implemented)")
except AssertionError as e:
    print(f"✗ FAILED: {e}")


## 3. `parse_yields(series)`

Convert mixed yield string forms to float percent in `[0, 100]`.

| Input      | Output |
|------------|--------|
| `'85%'`    | `85.0` |
| `'0.85'`   | `85.0` |
| `'85'`     | `85.0` |
| `'85.5%'`  | `85.5` |

**Trap:** `'0.85'` means 85%, not 0.85%. Assume any value `≤ 1.0` is a fraction; anything `> 1` is already a percent.

In [ ]:
def parse_yields(series: pd.Series) -> pd.Series:
    """Convert mixed yield string forms into float percent in [0, 100]."""
    # TODO: your implementation goes here
    raise NotImplementedError


In [ ]:
# Test cell — run to check your implementation
try:
    out = parse_yields(pd.Series(["85%", "0.85", "85", "85.5%"]))
    expected = pd.Series([85.0, 85.0, 85.0, 85.5])
    assert (out.round(1) == expected).all(), f"expected {list(expected)}, got {list(out)}"
    print("✓ passed")
except NotImplementedError:
    print("(not yet implemented)")
except AssertionError as e:
    print(f"✗ FAILED: {e}")


## 4. `standardize_temperature(series)`

Convert mixed-unit temperature strings to float degrees Celsius.

| Input               | Output              |
|---------------------|---------------------|
| `'80 C'`            | `80.0`              |
| `'353 K'`           | `79.85`             |
| `'80'` (no unit)    | `80.0` (assume °C)  |

In [ ]:
def standardize_temperature(series: pd.Series) -> pd.Series:
    """Convert mixed-unit temperature strings to float °C."""
    # TODO: your implementation goes here
    raise NotImplementedError


In [ ]:
# Test cell — run to check your implementation
try:
    out = standardize_temperature(pd.Series(["80 C", "353 K", "80"]))
    assert abs(out.iloc[0] - 80.0) < 0.01, f"first value should be 80, got {out.iloc[0]}"
    assert abs(out.iloc[1] - 79.85) < 0.02, f"second value should be 79.85, got {out.iloc[1]}"
    assert abs(out.iloc[2] - 80.0) < 0.01, f"third value should be 80, got {out.iloc[2]}"
    print("✓ passed")
except NotImplementedError:
    print("(not yet implemented)")
except AssertionError as e:
    print(f"✗ FAILED: {e}")


## 5. `build_tidy_reactions(messy_xlsx, catalyst_ref, solvent_ref)`

End-to-end: load the messy `.xlsx`, apply the four helpers above, and return a tidy `pd.DataFrame`.

**Output columns, in this exact order:**
1. `reaction_id`
2. `date`
3. `catalyst_canonical`
4. `solvent_canonical`
5. `temperature_C`
6. `time_min`
7. `yield_pct`
8. `operator`
9. `notes`

**Steps you'll typically do inside this function:**
- `pd.read_excel(messy_xlsx)` to load the sheet(s)
- Identify the reactions-log sheet (it's the one with rows of reactions)
- Rename messy column headers to your canonical names
- Apply `canonicalize_catalyst_names`, `canonicalize_solvent_names`, `parse_yields`, `standardize_temperature` to the appropriate columns
- Handle missing values (leave as NaN or `None`; don't put in strings like `'n/a'`)
- Return the DataFrame with columns in the required order

In [ ]:
def build_tidy_reactions(
    messy_xlsx: Path,
    catalyst_ref: pd.DataFrame,
    solvent_ref: pd.DataFrame,
) -> pd.DataFrame:
    """End-to-end: load messy xlsx, produce a tidy reactions_log DataFrame."""
    # TODO: your implementation goes here
    raise NotImplementedError


In [ ]:
# Test cell — end-to-end on the sprint dataset
try:
    tidy = build_tidy_reactions(_find("sprint-dataset-messy.xlsx"),
                                 CATALYST_REF, SOLVENT_REF)
    expected_cols = [
        "reaction_id", "date", "catalyst_canonical", "solvent_canonical",
        "temperature_C", "time_min", "yield_pct", "operator", "notes",
    ]
    assert list(tidy.columns) == expected_cols, f"columns mismatch: {list(tidy.columns)}"

    # Compare to the gold-standard tidy CSV
    gold = pd.read_csv(_find("reactions_log_tidy.csv"))
    tidy_sorted = tidy.sort_values("reaction_id").reset_index(drop=True)
    gold_sorted = gold.sort_values("reaction_id").reset_index(drop=True)
    pd.testing.assert_frame_equal(
        tidy_sorted[gold.columns], gold_sorted, check_dtype=False, atol=0.5
    )
    print("✓ passed — your pipeline reproduces the gold-standard tidy log")
except NotImplementedError:
    print("(not yet implemented)")
except AssertionError as e:
    print(f"✗ FAILED: {e}")
except FileNotFoundError as e:
    print(f"✗ FAILED: {e}")


## 6. `main()` — the CLI entry point

Read the messy `.xlsx`, produce three tidy CSVs next to the input:
- `reactions_log_tidy.csv`
- `catalyst_reference_tidy.csv`
- `solvent_reference_tidy.csv`

**Practical:** allow the messy xlsx path to be a command-line argument, or hardcoded to the sprint dataset with an override. In this notebook context, we'll just call `main('sprint-dataset-messy.xlsx')` to demo.

In [ ]:
def main(messy_xlsx_path: str = "sprint-dataset-messy.xlsx") -> None:
    """Read the messy xlsx, write three tidy CSVs next to it."""
    # TODO: your implementation goes here
    raise NotImplementedError


In [ ]:
# Demo run — writes CSVs to the current folder (or /content/ on Colab)
try:
    main("sprint-dataset-messy.xlsx")
    print("✓ passed — three tidy CSVs written")
except NotImplementedError:
    print("(not yet implemented)")


## 7. The generalization test — the important one

Anyone can write a script that cleans one specific file. The stretch-goal question is whether your pipeline **generalizes**.

Run the cell below on the second messy dataset — the one you haven't seen. If it passes, you have built a reusable asset.

In [ ]:
# Generalization test — run on new-experiments-messy.xlsx
try:
    tidy2 = build_tidy_reactions(_find("new-experiments-messy.xlsx"),
                                  CATALYST_REF, SOLVENT_REF)

    # Same schema
    expected_cols = [
        "reaction_id", "date", "catalyst_canonical", "solvent_canonical",
        "temperature_C", "time_min", "yield_pct", "operator", "notes",
    ]
    assert list(tidy2.columns) == expected_cols, \
        f"columns mismatch: {list(tidy2.columns)}"

    # All foreign keys resolve
    assert set(tidy2["catalyst_canonical"]) <= set(CATALYST_REF["catalyst_canonical"]), \
        "unknown catalyst survived canonicalization"
    assert set(tidy2["solvent_canonical"]) <= set(SOLVENT_REF["solvent_canonical"]), \
        "unknown solvent survived canonicalization"

    # Yields in valid range
    assert tidy2["yield_pct"].between(0, 100).all(), "yields out of [0, 100]"

    print("✓ passed — your pipeline generalizes to a second dataset")
    print(f"\n  {len(tidy2)} tidy rows extracted from the new file")
    display(tidy2.head())
except NotImplementedError:
    print("(build_tidy_reactions not yet implemented)")
except (AssertionError, FileNotFoundError) as e:
    print(f"✗ FAILED: {e}")


## Done. What you just built.

If the last test passed, you have built the **code leg of the data stewardship triad** in miniature:

1. **Data** — the three tidy CSVs your pipeline produces
2. **Code** — this pipeline, which can be handed to a colleague
3. **Docs** — the docstrings, the tests, and (bonus points) a data dictionary describing every output column

Anyone with your `pipeline_starter` code, the tidy reference CSVs, and a short README can now clean any log in this format without you sitting next to them. That's the entire DS lecture, compressed into a folder.

### If you want to keep going

- Add **data-quality assertions** inline (e.g. `assert tidy['temperature_C'].between(-10, 300).all()`) — catch bad rows on the next messy file BEFORE they become bugs downstream.
- Add **logging** — for each row processed, log which transformations applied. When someone complains "the yields look weird," the log tells you what happened.
- Write a **data dictionary** for your tidy output. You already saw the shape of one in the main tutorial — now write it for YOUR pipeline's output.
- **Package it** — add a `requirements.txt`, a README, put it in a git repo. Now anyone can install and run it.
